In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
import os, threading, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

BASE = ("https://data-cbr.csiro.au/thredds/fileServer/catch_all/oa-cmip6-wave/"
        "UniMelb-CSIRO_CMIP6_projections_phase1")
OUT_ROOT = Path(rf"{DATA_ROOT}/new_climate/criso_wind")

VAR    = "wnd"
CDFAC  = "CDFAC1"
MODELS = ["ACCESS-CM2", "EC-EARTH3"]

# remote scenario name -> local subdir (same convention as wave)  : (start YYYYMM, end YYYYMM)
SCENARIOS = {
    "historical": ("his", 198501, 201412),
    "ssp126":     ("126", 201501, 210012),
    "ssp585":     ("585", 201501, 210012),
}

MAX_WORKERS     = 4        # concurrency; drop to 2 if server is unstable
CHUNK           = 1 << 20  # 1 MB
VERIFY_EXISTING = False    # True: HEAD-check size of existing files (slow; enable once after first full run)

_stop  = threading.Event()
_local = threading.local()

def _session():
    s = getattr(_local, "s", None)
    if s is None:
        s = requests.Session()
        retry = Retry(total=5, backoff_factor=1.5,
                      status_forcelist=[429, 500, 502, 503, 504],
                      allowed_methods=["GET", "HEAD"])
        s.mount("https://", HTTPAdapter(max_retries=retry))
        _local.s = s
    return s

def months(a, b):
    y, m = divmod(a, 100)
    Y, M = divmod(b, 100)
    while (y, m) <= (Y, M):
        yield y * 100 + m
        m += 1
        if m == 13:
            y, m = y + 1, 1

def build_tasks():
    tasks = []
    for model in MODELS:
        for scen, (sub, a, b) in SCENARIOS.items():
            d = OUT_ROOT / model / sub
            d.mkdir(parents=True, exist_ok=True)
            for ym in months(a, b):
                fn = f"ww3.{ym}_{VAR}.nc"
                url = f"{BASE}/{scen}/{model}/{CDFAC}/ww3_ounf_glout/{fn}"
                tasks.append((url, d / fn))
    return tasks

def fetch(url, dest, bbar):
    """Download one file with resume. Returns 'skip' / 'done'; raises on failure."""
    part = Path(str(dest) + ".part")

    if dest.exists():
        if not VERIFY_EXISTING:
            return "skip"
        h = _session().head(url, timeout=60, allow_redirects=True)
        h.raise_for_status()
        remote = int(h.headers.get("Content-Length", 0))
        if remote and dest.stat().st_size == remote:
            return "skip"
        dest.unlink()          # size mismatch, re-download

    have = part.stat().st_size if part.exists() else 0
    headers = {"Range": f"bytes={have}-"} if have else {}
    r = _session().get(url, headers=headers, stream=True, timeout=(30, 600))

    if r.status_code == 416:               # already complete
        r.close()
        part.replace(dest)
        return "done"
    r.raise_for_status()

    if have and r.status_code == 206:
        cr = r.headers.get("Content-Range", "")
        total = int(cr.rsplit("/", 1)[-1]) if "/" in cr else None
        mode = "ab"
    else:                                   # server ignored Range, restart
        total = int(r.headers.get("Content-Length", 0)) or None
        mode, have = "wb", 0

    with r, open(part, mode) as f:
        for chunk in r.iter_content(CHUNK):
            if _stop.is_set():
                f.flush(); os.fsync(f.fileno())
                raise KeyboardInterrupt
            if chunk:
                f.write(chunk)
                bbar.update(len(chunk))

    size = part.stat().st_size
    if total and size != total:
        raise IOError(f"size mismatch: got {size}, expect {total}")
    part.replace(dest)         # rename only when complete -> any .nc in dir is intact
    return "done"


In [ ]:
tasks = build_tasks()
todo  = [t for t in tasks if not t[1].exists()]

free_gb = shutil.disk_usage(OUT_ROOT.drive + "\\").free / 1024**3
print(f"total {len(tasks)} files, {len(todo)} to download")
print(f"estimated ~{len(todo) * 0.083:.0f} GB needed, G: free {free_gb:.0f} GB")
for model in MODELS:
    for scen, (sub, a, b) in SCENARIOS.items():
        d = OUT_ROOT / model / sub
        n_all = sum(1 for _ in months(a, b))
        n_ok  = len(list(d.glob(f"ww3.*_{VAR}.nc")))
        print(f"  {model:12s} {sub:4s}  {n_ok:5d}/{n_all}")

_stop.clear()
failed = []
fbar = tqdm(total=len(todo), desc="files", unit="f")
bbar = tqdm(total=None, desc="bytes", unit="B", unit_scale=True, unit_divisor=1024)

try:
    with ThreadPoolExecutor(MAX_WORKERS) as ex:
        futs = {ex.submit(fetch, u, p, bbar): p for u, p in todo}
        for fu in as_completed(futs):
            p = futs[fu]
            try:
                fu.result()
            except KeyboardInterrupt:
                pass
            except Exception as e:
                failed.append((p.name, repr(e)))
                fbar.write(f"FAIL {p.parent.parent.name}/{p.parent.name}/{p.name}: {e}")
            fbar.update(1)
except KeyboardInterrupt:
    _stop.set()
    print("\nStop requested, waiting for threads... written .part files resume on next run")
finally:
    fbar.close(); bbar.close()

print(f"\nfailed {len(failed)}")
for n, e in failed[:20]:
    print(" ", n, e)


In [ ]:
missing = {}
for model in MODELS:
    for scen, (sub, a, b) in SCENARIOS.items():
        d = OUT_ROOT / model / sub
        miss = [ym for ym in months(a, b) if not (d / f"ww3.{ym}_{VAR}.nc").exists()]
        parts = list(d.glob("*.part"))
        print(f"{model:12s} {sub:4s}  missing {len(miss):4d}  leftover .part {len(parts)}")
        if miss:
            missing[(model, sub)] = miss
            print(f"    first/last: {miss[0]} .. {miss[-1]}")
